In [10]:
import pandas as pd
import matplotlib as mpl

import obspy
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy import Catalog

In [11]:
# client = Client('EMSC') # Client to query from
client = Client('IRIS') # Client to query from
# client = Client('ORFEUS') # Client to query from
#
# Time period to collect seismic events foe
t0 = UTCDateTime("2017-10-20T00:00:00.000")
t1 = UTCDateTime("2017-10-23T00:00:00.000")

In [12]:
events = client.get_events( t0,t1 )

Query events from EMSC: we need to split in shorter time spans for each query because of limitations on responce size

In [13]:
# events = Catalog()
# t=t0
# dt = 1*24*60*60
# while t < t1:
#     ev_period = client.get_events( t,t+dt )
#     events += ev_period
#     print(t, ", len = ", len(ev_period))
#     t+=dt

In [14]:
%matplotlib inline
mpl.rcParams['figure.figsize'] = (20.0, 10.0)
_=events.plot()

ImportError: Neither Basemap nor Cartopy could be imported.

Now create dataset that can be accessed from the visualization web client

In [7]:
pevents = []

for event in events:
    row = dict( event_type=event.event_type,
               magnitude = event.magnitudes[0]['mag'],
               time = pd.to_datetime( event.origins[0]['time'].isoformat() ),
               lon = event.origins[0]['longitude'],
               lat = event.origins[0]['latitude'],
               depth = event.origins[0]['depth']
              )
    pevents.append(row)
    
pevents = pd.DataFrame(pevents)

In [8]:
print(len(pevents))

906


In [9]:
import io
from pyarrow import feather
seismic_feather = r"earthquakes.feather"
pevents.to_feather(seismic_feather)